# Урок 2. Stacking.

Теперь поговорим о стекинге - другом популярном способе ансамблирования алгоритмов. Он принципиально отличается от бустинга двумя положениями. Первое, в стекинге обычно используются разнородные базовые алгоритмы (например, kNN, метод опорных векторов и дерево решений), тогда как в бустинге обычно используют ансамбли однородных алгоритмов (обычно применяются деревья решений). Второе, стекинг объединяет базовые алгоритмы, используя мета-алгоритм, который самостоятельно обучается на предоставленных базовыми моделями данных. Бустинг в свою очередь объединяет их, следуя детерминированному алгоритму.

Идея стекинга лежит на поверхности. Известно, что, если обучить несколько разных алгоритмов, то в задаче регрессии их среднее, а в задаче классификации — голосование по большинству, часто превосходят по качеству все эти алгоритмы по отдельности. Возникает вопрос: почему, собственно, нужно использовать для ансамблирования такие операции как усреднение или голосование? Можно же доверить ансамблироование очередному алгоритму (т.н. мета-алгоритму) машинного обучения, чтобы он сам решил, как ему действовать с полученными ответами базовых алгоритмов.

Чтобы понять, как работает классический стекинг, сначала разберем его простейшую реализацию - **блендинг (blending)**. Обучающую выборку делят на две части.

![](https://248006.selcdn.ru/public/DS.%20Block%202.%20M9/blending_1.png)

На первой обучают базовые алгоритмы. Затем получают их ответы на второй части и на тестовой выборке.

![](https://248006.selcdn.ru/public/DS.%20Block%202.%20M9/blending_2.png)

Понятно, что  ответ каждого алгоритма можно рассматривать как новый признак (т.н. метапризнак). На метапризнаках второй части обучения настраивают метаалгоритм. Затем запускают его на метапризнаках теста и получают ответ.

![](https://248006.selcdn.ru/public/DS.%20Block%202.%20M9/blending_3.png)

В такой реализации самый большой недостаток - деление обучающей выборки. Получается, что ни базовые алгоритмы, ни метаалгоритм не используют всего объёма обучения, а значит, мы можем пропустить важные данные.

Один из способов борьбы за использование всей обучающей выборки - реализация классического **стекинга (stacking)**. Понятно, что обучить базовые алгоритмы на всей обучающей выборке и потом для той же выборки построить метапризнаки нельзя: будет очевидное переобучение. Поэтому выборку разбивают на части (т.н. фолды, как в кросс-валидации).

![](https://248006.selcdn.ru/public/DS.%20Block%202.%20M9/meta_1.png)

Затем последовательно для каждого обучают базовые алгоритмы на всех фолдах, кроме одного, а на оставшемся получают ответы базовых алгоритмов и трактуют их как значения соответствующих метапризнаков на этом фолде.

![](https://248006.selcdn.ru/public/DS.%20Block%202.%20M9/meta_2.png)

Для получения метапризнаков объектов тестовой выборки базовые алгоритмы обучают на всей тренировочной выборке и берут их ответы на тестовой.

![](https://248006.selcdn.ru/public/DS.%20Block%202.%20M9/meta_3.png)

Для примера, если взять M базовых алгоритмов и обучить их на N объектах, то окончательная матрица метапризнаков для обучения мета-алгоритма будет выглядеть так:
![](https://248006.selcdn.ru/public/DS.%20Block%202.%20M9/meta_mtrx.png)

Отметим, что стекинг не всегда существенно повышает качество лучшего из базовых алгоритмов, однако если этот алгоритм убрать из базовых, то и качество стекинга не сильно падает.

Также возможно использовать стекинг с несколькими слоями, т.н. многоуровневый стекинг, т.е. ввести понятие мета-мета признака и мета-мета алгоритма и далее при желании расширить еще на несколько слоев. Такой метод часто применяется в соревнованиях по машинному обучению, где борьба идет за десятитысячные доли к точности модели, однако саму модель это сделает неинтерпретируемой, что плохо для бизнес-задач.

Вообще, в sklearn нет реализации стекинга, но существуют такие небазовые пакеты python как mlxtend или vecstack, в которых присутствует этот метод, которым можно воспользоваться "из коробки".

Мы же на примере базовой реализации блендинга покажем, как работает этот метод. В качестве базовых алгоритмов возьмем kNN, LogisticRegression, DesisionTreeClassifier и SupportVectorClassification, а в качестве мета-алгоритма будем использовать XGBoost.

Будем проверять работу стекинга на том же преобразованном датасете Titanic.

In [1]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import pandas as pd
import numpy as np

In [2]:
titanic = pd.read_csv('./data/9.7_titanic.csv')
targets = titanic.Survived
data = titanic.drop(columns='Survived')

x_train, x_test, y_train, y_test = train_test_split(data,
                                                    targets,
                                                    train_size=0.8,
                                                    random_state=0)

Так как нам нужно и обучить базовые алгоритмы на тренировочном сете, и сделать на этих же данных предсказания для обучения мета-алгоритма, разделим тренировочные данные еще на два датасета: train и valid.

In [5]:
train, valid, train_true, valid_true = train_test_split(x_train,
                                                        y_train,
                                                        train_size=0.5,
                                                        random_state=0)
valid.shape[0]

356

Обучим базовые алгоритмы.

In [4]:
knn = KNeighborsClassifier(n_neighbors=3)
knn_model = knn.fit(train, train_true)

lr = LogisticRegression(random_state=17)
lr_model = lr.fit(train, train_true)

dtc = DecisionTreeClassifier(max_leaf_nodes=4, random_state=17)
dtc_model = lr.fit(train, train_true)

svc = SVC(random_state=17)
svc_model = svc.fit(train, train_true)

C:\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
C:\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_resu

Теперь получим предсказания моделей для второй части тренировочных данных - valid, заполним получившимися метапризнаками матрицу для обучения мета-алгоритма и обучим его.

Заодно получим для каждого базового алгоритма метрику AUC для тестовых данных, которая покажет качество работы каждого алгоритма в отдельности.

In [6]:
models = [knn_model, lr_model, dtc_model, svc_model]
meta_mtrx = np.empty((valid.shape[0], len(models))) # (кол-во объектов, 4 алгоритма)
print(meta_mtrx)

for n, model in enumerate(models):
    meta_mtrx[:, n] = model.predict(valid)
    predicted = model.predict(x_test)
    print(f'{n} auc: {roc_auc_score(y_test, predicted)}')

meta = XGBClassifier(n_estimators=40)
meta_model = meta.fit(meta_mtrx, valid_true)

[[1.28712140e-311 1.28774986e-311 6.31465302e-320             nan]
 [2.10897680e-052 7.69110609e+218 1.29986551e+219 4.63461863e+228]
 [6.01681734e-154 2.19343361e-152 3.08012909e+175 4.26639590e+180]
 ...
 [1.80608275e-153 1.42644423e-153 3.61546017e-043 6.01347002e-154]
 [4.56335201e-072 6.37262397e-067 6.01347002e-154 3.01430316e-153]
 [6.01356718e-154 6.03823404e-154 4.66413009e-086 4.56335201e-072]]
0 auc: 0.7080368906455863
1 auc: 0.8094861660079051
2 auc: 0.8094861660079051
3 auc: 0.6538866930171279


Получим метапризнаки базовых алгоритмов для тестовых данных, чтобы мета-алгоритм мог по ним сделать предсказания.

In [7]:
meta_mtrx_test = np.empty((x_test.shape[0], len(models)))

for n, model in enumerate(models):
    meta_mtrx_test[:, n] = model.predict(x_test)

meta_predict = meta.predict(meta_mtrx_test)
print(f'Stacking AUC: {roc_auc_score(y_test, meta_predict)}')

Stacking AUC: 0.8094861660079051


Как мы видим, в этом случае финальный скор AUC не превышает результат лучших базовых алгоритмов. Попробуем убрать базовые алгоритмы с лучшими результатами и посмотреть, что изменится.

In [8]:
models = [knn_model, svc_model]
meta_mtrx = np.empty((valid.shape[0], len(models))) # (кол-во объектов, 4 алгоритма)

for n, model in enumerate(models):
    meta_mtrx[:, n] = model.predict(valid)
    predicted = model.predict(x_test)
    print(f'{n} auc: {roc_auc_score(y_test, predicted)}')

meta_model = meta.fit(meta_mtrx, valid_true)

meta_mtrx_test = np.empty((x_test.shape[0], len(models)))

for n, model in enumerate(models):
    meta_mtrx_test[:, n] = model.predict(x_test)

meta_predict = meta.predict(meta_mtrx_test)
print(f'Stacking AUC: {roc_auc_score(y_test, meta_predict)}')

0 auc: 0.7080368906455863
1 auc: 0.6538866930171279
Stacking AUC: 0.6538866930171279


Здесь же видно, что стекинг двух не самых лучших базовых алгоритмов дает результат выше, чем у каждого из них по отдельности.